In [2]:
from pathlib import Path
import numpy as np
import scipy.io as sio
import h5py


FILE = Path(
    "/projects/EEG-foundation-model/RECH202/"
    "speech_raw_datasets/BCI2020 EEG Signal for Words/"
    "Training set/Data_Sample01.mat"
)


def print_structure(obj, prefix=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else k

            try:
                arr = np.asarray(v)
                print(
                    f"{key:<40} "
                    f"type={type(v).__name__:<15} "
                    f"shape={arr.shape!s:<20} "
                    f"dtype={arr.dtype}"
                )
            except Exception:
                print(f"{key:<40} type={type(v).__name__}")

            print_structure(v, key)


def load_mat(path):
    try:
        return sio.loadmat(path, simplify_cells=True)

    except NotImplementedError:
        out = {}

        def read_h5(obj):
            if isinstance(obj, h5py.Dataset):
                arr = np.array(obj)
                return arr.T if arr.ndim >= 2 else arr

            if isinstance(obj, h5py.Group):
                return {k: read_h5(v) for k, v in obj.items()}

            return obj

        with h5py.File(path, "r") as f:
            for key in f.keys():
                out[key] = read_h5(f[key])

        return out


mat = load_mat(FILE)

print("\n========== STRUCTURE ==========\n")
print_structure(mat)

print("\n========== CONTENT PREVIEW ==========\n")

def preview_arrays(obj, prefix=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else k
            preview_arrays(v, key)

    elif isinstance(obj, np.ndarray):
        print(f"\n--- {prefix} ---")
        print("shape:", obj.shape)
        print("dtype:", obj.dtype)

        flat = obj.flatten()

        if flat.size > 0:
            print("first values:", flat[:10])


preview_arrays(mat)


========== STRUCTURE ==========

__header__                               type=bytes           shape=()                   dtype=|S76
__version__                              type=str             shape=()                   dtype=<U3
__globals__                              type=list            shape=(0,)                 dtype=float64
epo_train                                type=dict            shape=()                   dtype=object
epo_train.clab                           type=ndarray         shape=(64,)                dtype=object
epo_train.fs                             type=int             shape=()                   dtype=int64
epo_train.title                          type=str             shape=()                   dtype=<U4
epo_train.file                           type=str             shape=()                   dtype=<U4
epo_train.x                              type=ndarray         shape=(795, 64, 300)       dtype=float64
epo_train.y                              type=ndarray     

In [3]:
from pathlib import Path
import numpy as np
import scipy.io as sio
import h5py


ROOT = Path(
    "/projects/EEG-foundation-model/RECH202/"
    "speech_raw_datasets/BCI2020 EEG Signal for Words"
)

FILES = [
    ROOT / "Training set" / "Data_Sample01.mat",
    ROOT / "Validation set" / "Data_Sample01.mat",
    ROOT / "Test set" / "Data_Sample01.mat",
]


def load_mat(path):
    try:
        return sio.loadmat(path, simplify_cells=True)
    except NotImplementedError:
        out = {}

        def read_h5(obj):
            if isinstance(obj, h5py.Dataset):
                arr = np.array(obj)
                return arr.T if arr.ndim >= 2 else arr

            if isinstance(obj, h5py.Group):
                return {k: read_h5(v) for k, v in obj.items()}

            return obj

        with h5py.File(path, "r") as f:
            for key in f.keys():
                out[key] = read_h5(f[key])

        return out


def get_epo(mat):
    for key in ["epo_train", "epo_test", "epo_val", "epo_validation"]:
        if key in mat:
            return key, mat[key]

    candidates = [
        k for k in mat.keys()
        if k.startswith("epo") and isinstance(mat[k], dict)
    ]

    if len(candidates) == 1:
        return candidates[0], mat[candidates[0]]

    raise KeyError(f"No epo_* found. Available keys: {list(mat.keys())}")


for file in FILES:
    print("\n" + "=" * 100)
    print("FILE:", file)
    print("=" * 100)

    mat = load_mat(file)

    print("\nTOP KEYS:")
    print([k for k in mat.keys() if not k.startswith("__")])

    epo_key, epo = get_epo(mat)

    print("\nEPO KEY:", epo_key)
    print("EPO KEYS:", list(epo.keys()))

    print("\nFIELDS:")

    for k, v in epo.items():
        arr = np.asarray(v)
        print(
            f"{k:<15} "
            f"type={type(v).__name__:<12} "
            f"shape={arr.shape!s:<20} "
            f"dtype={arr.dtype}"
        )

    print("\nCORE CHECK:")

    x = np.asarray(epo["x"])
    print("x shape:", x.shape)

    if x.ndim == 3:
        n_times, n_channels, n_trials = x.shape
        print("n_times:", n_times)
        print("n_channels:", n_channels)
        print("n_trials:", n_trials)

    if "fs" in epo:
        fs = float(np.asarray(epo["fs"]).squeeze())
        print("fs:", fs)
        print("trial duration sec:", n_times / fs)

    if "y" in epo:
        y = np.asarray(epo["y"]).squeeze()
        print("y shape:", y.shape)
        print("y dtype:", y.dtype)

        if y.ndim == 2:
            if y.shape[1] == n_trials:
                labels = np.argmax(y, axis=0)
                print("label format: one-hot/class x trials")
            elif y.shape[0] == n_trials:
                labels = np.argmax(y, axis=1)
                print("label format: trials x class")
            else:
                labels = None
                print("label format: cannot align with trials")

        elif y.ndim == 1:
            labels = y
            print("label format: vector")

        else:
            labels = None
            print("label format: unexpected")

        if labels is not None:
            print("labels shape:", labels.shape)
            print("unique labels:", np.unique(labels))
            print("first 20 labels:", labels[:20])
    else:
        print("No y field found")

    if "className" in epo:
        print("className:", np.asarray(epo["className"]).squeeze())

    if "clab" in epo:
        clab = np.asarray(epo["clab"], dtype=object).squeeze()
        print("clab shape:", clab.shape)
        print("clab first 10:", clab.flatten()[:10])


FILE: /projects/EEG-foundation-model/RECH202/speech_raw_datasets/BCI2020 EEG Signal for Words/Training set/Data_Sample01.mat

TOP KEYS:
['epo_train', 'mnt']

EPO KEY: epo_train
EPO KEYS: ['clab', 'fs', 'title', 'file', 'x', 'y', 't', 'className']

FIELDS:
clab            type=ndarray      shape=(64,)                dtype=object
fs              type=int          shape=()                   dtype=int64
title           type=str          shape=()                   dtype=<U4
file            type=str          shape=()                   dtype=<U4
x               type=ndarray      shape=(795, 64, 300)       dtype=float64
y               type=ndarray      shape=(5, 300)             dtype=uint8
t               type=ndarray      shape=(795,)               dtype=float64
className       type=ndarray      shape=(5,)                 dtype=object

CORE CHECK:
x shape: (795, 64, 300)
n_times: 795
n_channels: 64
n_trials: 300
fs: 256.0
trial duration sec: 3.10546875
y shape: (5, 300)
y dtype: uint8
labe